# 21 · Paper 1 v2: consolidate the 24 runs (CPU only, ~5 min, read-only on run folders)
Writes one slim table per evaluation set to `MyDrive/P1_v2/consolidated/`. Nothing in `runs/` is changed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, pandas as pd
OUT = '/content/drive/MyDrive/P1_v2'; C = OUT + '/consolidated'; os.makedirs(C, exist_ok=True)
dirs = sorted(d for d in glob.glob(OUT + '/runs/*_seed*') if os.path.isdir(d))
print(len(dirs), 'run folders')
missing = [(os.path.basename(d), f) for d in dirs for f in ('run.json','test_per_image.csv','dmid_per_image.csv','mias_per_image.csv','history.csv') if not os.path.exists(f'{d}/{f}')]
print('missing files:', missing)
assert len(dirs) == 24 and not missing, 'STOP: expected 24 complete run folders'

In [ ]:
def tag(df, d):
    name = os.path.basename(d); exp, m, s = name.split('_')
    df = df.drop(columns=[c for c in ('run','exp','model','seed') if c in df.columns])
    df.insert(0, 'run', name); df.insert(1, 'exp', exp); df.insert(2, 'model', m); df.insert(3, 'seed', int(s.replace('seed','')))
    return df
T, E, M, H, R = [], [], [], [], []
for d in dirs:
    T.append(tag(pd.read_csv(d + '/test_per_image.csv'), d))
    E.append(tag(pd.read_csv(d + '/dmid_per_image.csv'), d))
    m = pd.read_csv(d + '/mias_per_image.csv'); n0 = len(m)
    m = m.drop_duplicates('file_id', keep='first')
    M.append(tag(m, d)); print(os.path.basename(d), 'MIAS', n0, '->', len(m))
    h = pd.read_csv(d + '/history.csv'); H.append(tag(h, d))
    r = json.load(open(d + '/run.json')); r = {k: v for k, v in r.items() if not isinstance(v, (list, dict))}
    R.append(tag(pd.DataFrame([r]), d))
T, E, M, H, R = (pd.concat(x, ignore_index=True) for x in (T, E, M, H, R))
for nm, df in [('test', T), ('dmid', E), ('mias', M), ('history', H), ('runs', R)]:
    df.to_csv(f'{C}/p1v2_{nm}.csv.gz', index=False); print(nm, df.shape)
print(T.groupby(['exp','model']).size())
print(M.groupby('run').file_id.nunique().describe())

In [ ]:
# summary for a quick look (not the analysis)
cols = [c for c in ['dice'] if c in T.columns]
print(T.groupby(['exp','model'])[cols].mean().round(4))
print(E.groupby(['exp','model'])[cols].mean().round(4))
print(M.groupby(['exp','model'])[['dice_expert','dice_expert_breast','dice_proxy']].mean().round(4))
print(T.columns.tolist()); print(R.columns.tolist())
print('DONE')